### Imports

Initial package imports for data manipulation and numerical operations.

In [1]:
# Imports
import pandas as pd
import numpy as np

### Display Settings

Configured pandas to display all columns in output.

In [2]:
# Display settings
pd.set_option('display.max_columns', None)

### Data Loading

Loaded the loan data from parquet file.

In [3]:
# Data loading
loans = pd.read_parquet('../data/interim/parquet/accepted.parquet')

Created a working copy of the dataset to preserve original data.

In [4]:
# Create a copy
clean = loans.copy()

Displayed first few rows of the dataset to inspect its structure.

In [5]:
# Checking the data
clean.head()

,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,purpose,title,zip_code,addr_state,dti,earliest_cr_line,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,application_type,mort_acc,pub_rec_bankruptcies
0,32000.0,60 months,10.49,687.65,B,B3,Public Service,10+ years,MORTGAGE,120000.0,Verified,Feb-2015,Current,debt_consolidation,Debt consolidation,919xx,CA,24.05,Oct-1981,20.0,0.0,39687.0,57.8,42.0,w,Individual,2.0,0.0
1,9600.0,36 months,12.99,323.42,C,C1,None,None,RENT,21900.0,Verified,May-2014,Fully Paid,debt_consolidation,Debt consolidation,331xx,FL,10.03,Apr-2001,13.0,1.0,4509.0,38.9,20.0,w,Individual,0.0,1.0
2,4000.0,36 months,6.68,122.93,A,A3,System Analyst,4 years,MORTGAGE,83000.0,Not Verified,Apr-2015,Fully Paid,major_purchase,Major purchase,333xx,FL,19.53,Sep-2003,16.0,0.0,1564.0,17.2,25.0,w,Individual,2.0,0.0
3,6025.0,36 months,10.91,197.00,B,B4,Admin assistant,10+ years,RENT,52000.0,Not Verified,Dec-2017,Fully Paid,debt_consolidation,Debt consolidation,021xx,MA,9.16,Jun-2005,11.0,0.0,2706.0,12.8,25.0,w,Individual,0.0,0.0
4,25000.0,60 months,26.30,752.96,E,E5,Coordinator,10+ years,OWN,65000.0,Verified,Feb-2018,Current,debt_consolidation,Debt consolidation,926xx,CA,36.26,Jul-1999,19.0,0.0,49461.0,24.7,33.0,w,Individual,0.0,0.0


### Type Casting & Formatting

Standardized data types and formats:
- Converted interest rates from string with '%' to float
- Transformed loan terms from string (e.g., '36 months') to numeric values
- Standardized employment length by:
  - Converting text descriptions to numeric values
  - Mapping '10+ years' to 10 and '< 1 year' to 0
  - Handling missing values

In [6]:
# Convert datatypes and standardize formats
# Convert interest rate to float (remove % if present)
clean['int_rate'] = clean['int_rate'].astype(str).str.replace('%', '').astype(float)

# Convert term to numeric, handling None/NaN values
clean['term'] = (clean['term']
                .replace({None: np.nan})  
                .str.replace(' months', '')
                .astype(float)  
                .astype('Int64'))

# Clean emp_length column
clean['emp_length'] = (clean['emp_length']
                      .replace('None', np.nan)  
                      .replace('10+ years', '10') 
                      .replace('< 1 year', '0')
                      .str.replace(' years', '')  
                      .str.replace(' year', '')   
                      .astype(float)
                      .astype('Int64'))

### Missing Value Handling

Handled missing values using imputation:
- Numeric columns: Filled with median values to maintain data distribution
- Categorical columns: Filled with mode (most frequent value)

This approach preserved the overall data distribution while handling missing values effectively.

In [7]:
# Handle missing values 
# For numeric columns, fill with median
numeric_cols = ['annual_inc', 'dti', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
                'mort_acc', 'pub_rec_bankruptcies', 'loan_amnt', 'installment', 'int_rate', 'term']

for col in numeric_cols:
    clean[col] = clean[col].fillna(clean[col].median())

# For categorical columns, fill with mode
categorical_cols = ['emp_title', 'emp_length', 'verification_status', 'title',
                    'grade', 'sub_grade', 'home_ownership', 'purpose']
    
for col in categorical_cols:
    clean[col] = clean[col].fillna(clean[col].mode()[0])

### Date Formatting

Standardized date columns to datetime format (yyyy-mm-dd).

In [8]:
# Convert dates to datetime (yyyy-mm-dd)
clean['issue_d'] = pd.to_datetime(clean['issue_d'], format='%b-%Y')
clean['earliest_cr_line'] = pd.to_datetime(clean['earliest_cr_line'], format='%b-%Y')

### Outlier Handling

Handled outliers in key financial metrics:
- Capped Debt-to-Income ratio (DTI) at 99th percentile
- Capped revolving utilization at 99th percentile

This removed extreme values while preserving the majority of the data distribution.

In [9]:
# Handle outliers
# Cap DTI
clean['dti'] = clean['dti'].clip(upper=clean['dti'].quantile(0.99))

# Cap revolving utilization
clean['revol_util'] = clean['revol_util'].clip(upper=clean['revol_util'].quantile(0.99))

### Column Dropping

Dropped unnecessary columns to simplify the dataset:
- 'purpose' column was removed as it was not needed for the analysis

In [10]:
# Drop unnecessary columns
cols_to_drop = ['purpose'] 
clean = clean.drop(columns=cols_to_drop)

### Export 

Saved the cleaned dataset.

In [ ]:
# Save to cleaned folder
clean.to_parquet('../data/cleaned/parquet/accepted.parquet', index=False)